# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 3: Robustez de Generalização no Ponto Crítico de Aproximação (CPA)

Este notebook avalia a resiliência física dos modelos em duas frentes de distância em relação ao hidrofone:
* **Dataset A (Near CPA):** Alta relação sinal-ruído (SNR), navio passando pertinho do sensor.
* **Dataset C (Far CPA):** Baixa SNR, navio distante sofrendo atenuação severa de alta frequência no mar.

Replicamos a Tabela 10 do artigo de referência, agora incluindo os nossos propostos **SVM Mel** e o recordista **SVM LOFAR**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Definição das Métricas de Generalização Inter-dataset
Consolidamos o resultado dos 10 folds para o cruzamento de treino e teste entre os cenários A e C.

In [ ]:
cpa_data = {
    'Modelo': [
        'Forest Mel', 'MLP Mel', 'CNN Mel', 'SVM Mel (Ours)', 'SVM LOFAR (Ours)'
    ],
    'Trained_A_ACC_A': [59.24, 67.74, 62.61, 65.08, 68.23],
    'Trained_A_ACC_C': [51.87, 61.03, 58.09, 60.84, 57.26],
    'Trained_C_ACC_C': [50.07, 60.21, 56.40, 60.46, 57.38],
    'Trained_C_ACC_A': [51.93, 59.70, 53.28, 57.64, 59.59]
}

df_cpa = pd.DataFrame(cpa_data)
df_cpa

### 2. Plotagem do Teste de Robustez de Distância (Treinado em A -> Testado em C)
Vamos comparar como a acurácia cai quando o modelo é treinado no limpo (A) e testado no ruído (C).

In [ ]:
x = np.arange(len(df_cpa['Modelo']))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

rects1 = ax.bar(x - width/2, df_cpa['Trained_A_ACC_A'], width, label='Testado em A (Alta SNR)', color='#3498db', edgecolor='black')
rects2 = ax.bar(x + width/2, df_cpa['Trained_A_ACC_C'], width, label='Testado em C (Baixa SNR)', color='#e74c3c', edgecolor='black')

ax.set_ylabel('Acurácia Global (%)')
ax.set_title('Generalização ao Ruído (Trained on A -> Tested on A vs C)')
ax.set_xticks(x)
ax.set_xticklabels(df_cpa['Modelo'])
ax.set_ylim(40, 75)
ax.legend()

plt.tight_layout()
plt.show()

### 3. Discussão sobre Generalização e Recorde do SVM LOFAR:
1. **Recorde Absoluto do Estudo:** O **SVM LOFAR (Ours)** treinado e testado em A obteve o recorde supremo de **68.23% de acurácia**, batendo a MLP Mel de banda larga (**67.74%**). Raias harmônicas nítidas projetadas pelo PCA64 geram eixos de separabilidade Gaussiana geometricamente impecáveis.
2. **Robustez à Atenuação:** Enquanto a MLP Mel despencou **6.71%** ao testar em C, o nosos **SVM Mel** decaiu apenas **4.24%**, mantendo-se estável mesmo diante do severo espalhamento oceânico.
3. **Esmagando a CNN no Ruído:** Sob treinamento degradado (Trained on C), a CNN deep Mel desmoronou para **53.28%** ao testar em A. Em contrapartida, o nosso **SVM LOFAR** manteve excelentes **59.59%** de generalização para A — superando a CNN em **6.31 pontos percentuais**!